# OpenBB Tools

## Getting Started

### Install dependencies

In [1]:
# !pip install openbb
# !pip install openbb-charting

# !pip install pydantic==2.5

# !pip install langchain
# !pip install langchain_community
# !pip install langchain_openai
# need to restart runtime after installing pydantic

### Set the OpenAI API Key

In [2]:
import os
from pathlib import Path
from dotenv import load_dotenv

candidates = [Path.cwd() / ".env", Path.cwd().parent / ".env"]
loaded = False
for env_path in candidates:
    if env_path.exists():
        load_dotenv(env_path, override=False)
        loaded = True
        break

if not loaded:
    raise FileNotFoundError(".env file not found in current or parent directory")

if not os.getenv("AZURE_OPENAI_API_KEY"):
    raise ValueError("AZURE_OPENAI_API_KEY not found in loaded .env")

print("Azure OpenAI credentials loaded from .env")
print(f"  Endpoint: {os.getenv('AZURE_OPENAI_ENDPOINT')}")
print(f"  Deployment: {os.getenv('AZURE_OPENAI_DEPLOYMENT')}")

Azure OpenAI credentials loaded from .env
  Endpoint: https://mastersaifoundry.openai.azure.com/
  Deployment: gpt-4o


### Import packages

In [3]:
from openbb import obb

## OpenBB tool pre-requisites

### Callable

In [19]:
obb.equity.price.historical(symbol="AAPL", provider="fmp_cached").to_df()

,open,high,low,close,volume,vwap,change,change_percent,symbol,dividend
date,,,,,,,,,,
2025-03-03,241.79,244.03,236.11,238.03,47184000,239.9900,-3.760,-0.015600,AAPL,NaN
2025-03-04,237.71,240.07,234.68,235.93,53798100,237.0975,-1.780,-0.007488,AAPL,NaN
2025-03-05,235.42,236.55,229.23,235.74,47227643,234.2350,0.320,0.001359,AAPL,NaN
2025-03-06,234.44,237.86,233.16,235.33,45170419,235.1975,0.895,0.003796,AAPL,NaN
2025-03-07,235.11,241.37,234.76,239.07,46273600,237.5775,3.970,0.016800,AAPL,NaN
...,...,...,...,...,...,...,...,...,...,...
2026-02-25,271.78,274.94,271.05,274.23,33714342,273.0000,2.450,0.009015,AAPL,NaN
2026-02-26,274.95,276.11,270.80,272.95,32345114,273.7025,-2.000,-0.007274,AAPL,NaN
2026-02-27,272.81,272.81,262.89,264.18,72366505,268.1725,-8.630,-0.031600,AAPL,NaN


In [20]:
obb.equity.price.historical(
    "AAPL", start_date="2022-01-01", provider="fmp_cached"
).charting.to_chart()

### Input Schema

In [21]:
obb.coverage.command_model[".equity.price.historical"]["openbb"]["QueryParams"]

{'fields': {'symbol': FieldInfo(annotation=str, required=True, description='Symbol to get data for.', json_schema_extra={'fmp': {'multiple_items_allowed': True}, 'fmp_cached': {'multiple_items_allowed': True, 'choices': None}, 'yfinance': {'multiple_items_allowed': True}}),
  'start_date': FieldInfo(annotation=Union[date, NoneType], required=False, default=None, description='Start date of the data, in YYYY-MM-DD format.'),
  'end_date': FieldInfo(annotation=Union[date, NoneType], required=False, default=None, description='End date of the data, in YYYY-MM-DD format.')},
 'docstring': 'Equity Historical Price Query.'}

In [22]:
obb.coverage.command_model[".equity.price.historical"]["fmp_cached"]["QueryParams"]

{'fields': {'interval': FieldInfo(annotation=Literal['1m', '5m', '15m', '30m', '1h', '4h', '1d'], required=False, default='1d', description='Time interval of the data to return.'),
  'adjustment': FieldInfo(annotation=Literal['splits_only', 'splits_and_dividends', 'unadjusted'], required=False, default='splits_only', description="The adjustment type for the data. 'splits_only' is adjusted for splits only. 'splits_and_dividends' is adjusted for both splits and dividends. 'unadjusted' is the raw, unadjusted data."),
  'include_dividends': FieldInfo(annotation=bool, required=False, default=True, description='Include dividend data in the results. When True, fetches dividend information from FMP and merges it with price data.')},
 'docstring': 'FMP Cached Equity Historical Query Parameters.\n    \n    Independent query parameters for the cached provider.\n    '}

### Documentation

In [23]:
help(obb.equity.price.historical)

Help on method historical in module openbb.package.equity_price:

historical(symbol: Annotated[str | list[str], OpenBBField(description='Symbol to get data for. Multiple comma separateditems allowed for provider(s): fmp, fmp_cached,yfinance.')], start_date: Annotated[datetime.date | None | str, OpenBBField(description='Start date of the data, in YYYY-MM-DD format.')] = None, end_date: Annotated[datetime.date | None | str, OpenBBField(description='End date of the data, in YYYY-MM-DD format.')] = None, chart: Annotated[bool, OpenBBField(description='Whether to create a chart or not, by defaultFalse.')] = False, provider: Annotated[Optional[Literal['fmp', 'fmp_cached', 'yfinance']], OpenBBField(description='The provider to use, by default None. If None, thepriority list configured in the settings is used.Default priority: fmp, fmp_cached, yfinance.')] = None, **kwargs) -> openbb_core.app.model.obbject.OBBject method of openbb.package.equity_price.ROUTER_equity_price instance
    Get histo

## OpenBB tool

In [24]:
from langchain_core.tools import StructuredTool

llm_historical_price = StructuredTool.from_function(
    func=obb.equity.price.historical,
    description=obb.equity.price.historical.__doc__.split("\n")[
        0
    ],  # Use first line of docstring
)

## Multiple OpenBB Tools

In [25]:
llm_tools = [
    StructuredTool.from_function(
        name=name,
        func=schema["callable"],
        description=schema["callable"].__doc__.split("\n")[0],
    )
    for name, schema in obb.coverage.command_schemas().items()
]

## OpenBB Tools fed to agent

In [6]:
from langchain.agents import create_agent
from langchain_openai import AzureChatOpenAI
from langchain_core.tools import StructuredTool


def equity_price_quote(symbol: str) -> str:
    """Get the latest stock price quote for a given ticker symbol."""
    return obb.equity.price.quote(symbol=symbol, provider="fmp_cached")


llm_tools = [
    StructuredTool.from_function(
        func=equity_price_quote,
        name="equity_price_quote",
        description="Get the latest stock price quote for a given ticker symbol.",
    )
]

llm = AzureChatOpenAI(
    azure_deployment=os.getenv("AZURE_OPENAI_DEPLOYMENT", "gpt-4o"),
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    api_version=os.getenv("AZURE_OPENAI_API_VERSION", "2024-12-01-preview"),
    temperature=0.1,
)

agent = create_agent(
    model=llm,
    tools=llm_tools,
    system_prompt="You are a very powerful assistant, but don't know current events",
)

result = agent.invoke(
    {"messages": [{"role": "user", "content": "What is the latest stock price of AAPL?"}]}
)

for msg in result["messages"]:
    print(f"{msg.type}: {msg.content}")

human: What is the latest stock price of AAPL?
ai: 
tool: id='069a8a79-1bdd-76fe-8000-a168daa72e8e' results=[FMPEquityQuoteData(symbol=AAPL, asset_type=None, name=Apple Inc., exchange=NASDAQ, bid=None, bid_size=None, bid_exchange=None, ask=None, ask_size=None, ask_exchange=None, quote_conditions=None, quote_indicators=None, sales_conditions=None, sequence_number=None, market_center=None, participant_timestamp=None, trf_timestamp=None, sip_timestamp=None, last_price=262.52, last_tick=None, last_size=None, last_timestamp=2026-03-04 21:00:01, open=264.65, high=266.15, low=261.42, close=None, volume=39258957, exchange_volume=None, prev_close=263.75, change=-1.23, change_percent=-0.0046635, year_high=288.62, year_low=169.21, ma50=265.019, ma200=243.43234, market_cap=3858499397272.0)] provider='fmp_cached' warnings=None chart=None extra={'metadata': Metadata

arguments: {'provider_choices': {'provider': 'fmp_cached'}, 'standard_params': {'symbol': 'AAPL'}, 'extra_params': {}}
duration: 85323

In [ ]:
import os
from pathlib import Path
import importlib.metadata as md
import openbb

workspace_root = Path(r"I:/masterswork/git/OpenBB").resolve()
openbb_file = Path(openbb.__file__).resolve()

dist = md.distribution("openbb-core")
direct_url = None
try:
    direct_url = dist.read_text("direct_url.json")
except Exception:
    direct_url = None

print("kernel_python:", os.sys.executable)
print("openbb_file:", openbb_file)
print("openbb_file_under_workspace:", str(openbb_file).startswith(str(workspace_root)))
print("openbb-core version:", md.version("openbb-core"))
print("openbb-core location:", Path(dist.locate_file("")))
print("has direct_url.json:", bool(direct_url))
if direct_url:
    print("direct_url.json:", direct_url)


kernel_python: i:\masterswork\git\OpenBB\.venv_win\Scripts\python.exe
openbb_file: I:\masterswork\git\OpenBB\openbb_platform\core\openbb\__init__.py
openbb_file_under_workspace: True
openbb-core version: 1.5.5
openbb-core location: i:\masterswork\git\OpenBB\.venv_win\Lib\site-packages
has direct_url.json: True
direct_url.json: {"dir_info": {"editable": true}, "url": "file:///I:/masterswork/git/OpenBB/openbb_platform/core"}
